In [ ]:
import sys
import praw
from praw.models import MoreComments
import pandas as pd
import numpy as np
import re
from urlextract import URLExtract 
from datetime import datetime, timedelta
from collections import Counter
from urlextract import URLExtract 
from tldextract import extract
from IPython.core.display import display, HTML
import matplotlib.pyplot as plt

In [ ]:
import sys
import pandas as pd
import numpy as np
import re
from datetime import datetime, timedelta
from urlextract import URLExtract 
from tldextract import extract
from IPython.core.display import display, HTML

In [ ]:
datetime.now()+timedelta(hours=2)

In [ ]:
covid_regex = '(coronav|covid|pandemi|epidemi|sars)'
fake_regex = '(fake |false ).{0,15}(new\\w|info\\w)'+'|'+\
             '(dis|mis|mal)(info\\w|leading)'+'|'+\
             '(unreli\\w+|unverifi\\w+|untrustw\\w+|fake\\w+)( source)'
fake_yes ='(th.. is|you are|stop spreading|^).{0,15}'+ fake_regex
fake_no = '\\/s$|\\\s$|\\bjk\\b|(not .{0,15})'+fake_regex

In [ ]:
def search_corona(text): return bool(re.search(covid_regex, str(text).lower()))
def search_fake(text): return bool(re.search(fake_regex, str(text).lower()))

def search_yes_no(yes, no, text):
    return bool(re.search(yes, str(text).lower())) and not bool(re.search(no, str(text).lower()))    
def search_yes(yes, text):
    return bool(re.search(yes, str(text).lower()))    
def search_no(no, text):
    return not bool(re.search(no, str(text).lower()))    

def has_url(text):
    return URLExtract().has_urls(text)
def extract_domain(url):
    tsd, td, tsu = extract(url)
    return(td + '.' + tsu)
def highlight_re(text, regex):
    htmld = re.sub(r'('+regex+')', '<mark>' + r'\1' + '</mark>', text.lower())
    display(HTML(htmld))

In [ ]:
def import_reddit(from_date = '2020-03-01', subreddits = 'all'):
    latest_time = datetime.now()-timedelta(hours=2)
    ###
    ALL = pd.read_csv('http://ec2-3-8-1-23.eu-west-2.compute.amazonaws.com/vlad/all_submissions.csv')
    SR = pd.read_csv("R_selected_subreddits.csv")
    if subreddits != 'all':
        SR = SR[SR.text.isin(list(subreddits))]
    
    ###
    S = pd.read_csv("http://ec2-3-8-1-23.eu-west-2.compute.amazonaws.com/vlad/R_all_submissions.csv")
    S.columns = S.columns.str.strip()
    
    ###
    C = pd.read_csv("http://ec2-3-8-1-23.eu-west-2.compute.amazonaws.com/vlad/R_all_comments.csv")
    C.columns = C.columns.str.strip()
    C['date_created'] = [datetime.fromtimestamp(int(c))-timedelta(hours = 8) for c in C.date_created]

    ###
    CORONA = \
    C.rename(columns = {'id':'comm_id', 'author_id':'comm_author_id','submission_id':'subm_id',\
                    'body':'comm_body', 'score':'comm_score','date_created':'comm_date'}).\
      set_index(['subm_id', 'comm_id']).\
    join(
        ALL.drop('author_id',1)\
           [(ALL.date_created>=from_date) & \
            (ALL.date_created<str(latest_time))].\
           sort_values(['subr_id','date_created']).\
           set_index('subr_id').\
         join(SR.\
               rename(columns = {'id':'subr_id', 'text':'subreddit'})\
               [['subr_id','subreddit']].\
               set_index('subr_id'),\
              on = 'subr_id', how = 'inner').\
              reset_index().set_index(['subr_id','subm_id']).\
         join(S.rename(columns = {'id':'subm_id','title':'subm_title','score':'subm_score','url':'subm_url'})\
              [['subr_id','subm_id','subm_title','subm_score','subm_url']].\
                set_index(['subr_id','subm_id']),\
              how = 'left', on = ['subr_id','subm_id']).\
         reset_index().\
         assign(subm_id = lambda df: 't3_'+ df.subm_id).\
         set_index('subm_id'),\
    on = 'subm_id', how = 'inner')
    
    ###
    print('IMPORTED!\nSubreddits: %d \nSubmissions: %d\nComments: %d' % 
          (SR.shape[0] ,len(set(CORONA.index.get_level_values(level = 0))), CORONA.shape[0]))
    return CORONA

In [ ]:
def print_corona_sc(CORONA_DF, n=0, sort = 'newest', rand = False, randseed=np.random.RandomState(), charlim = False):
    ### Filter out long comments
    if charlim>0:
        CORONA_DF = CORONA_DF[[len(b) < charlim for b in CORONA_DF['comm_body']]]
    ### Set n to the number of rows (if 0)
    if n==0: 
        n = CORONA_DF.shape[0]
    ### Sorting
    if sort == 'newest':
        CORONA_DF.sort_values('date_added',ascending = False,inplace = True)
    elif sort == 'most_flagged':
        submission_counts = CORONA_DF['subreddit'].count(level = 'subm_id').rename('flg_count')
        CORONA_DF.join(submission_counts, on ='subm_id').\
        sort_values(['flg_count','date_added'], ascending = [False, False] )
        submissions = pd.Series(CORONA_DF.index.get_level_values('subm_id')).drop_duplicates().tolist()
    ### Extract random sample
    if rand: 
        submissions = pd.Series(submissions).sample(n, random_state = randseed)
        CORONA_DF = CORONA_DF.loc[submissions]
    else:
        submissions = submissions[:n]
    ### Loop to print flagged submissions
    for i, t in enumerate(submissions):
        if i<=n:
            print('\n\n')
            D = CORONA_DF.loc[t]
            print('r/'+D.subreddit[0])
            highlight_re('<b>'+D.subm_title[0]+'</b>', covid_regex)
            print('URL: %s \nPublished: %s ' % (D.subm_url[0], D.date_created[0]))
            for c in D.iterrows():
                highlight_re('<p style="margin-left: 40px"> -> (score: %d) %s </p><br>' % 
                             (c[1].comm_score ,c[1].comm_body), fake_regex)

## 1. Importing Submissions and Comments

In [ ]:
COMM = import_reddit(from_date = '2020-04-01')

In [ ]:
max(COMM.date_added)

In [ ]:
def filter_comm(DF, subm_regex_yes, comm_regex_yes, subm_regex_no = 'noneXXXnone', comm_regex_no = 'noneXXXnone'):
    FILT = DF[np.logical_and(
                np.logical_and(
                  np.logical_and([search_yes(subm_regex_yes, s) for s in DF.subm_title],\
                                 [search_no(subm_regex_no, s) for s in DF.subm_title]),\
                  [search_yes(comm_regex_yes, c) for c in DF.comm_body ]),\
              [search_no(comm_regex_no, c) for c in DF.comm_body ])]
    return FILT

In [ ]:
covid_regex = '(coronav|covid|pandemi|epidemi|sars)'
fake_regex = '(fake |false).{0,15}(new\\w|info\\w|hood\\w)'+'|'+\
             '(dis|mis|mal)(info\\w|leading)'+'|'+\
             'bs|bullshit'+\
             '(not ).{0,5}(true\\b|verif\\w|real\\b)'+'|'+\
             '(unreli\\w+|unverifi\\w+|untrustw\\w+|fake\\w+)( source)'
fake_regex_no = '((no|not) .{0,10}('+fake_regex+'))'+'|(users often report submissions from this site)'
fake_regex_no

In [ ]:
CORONAFLAG = filter_comm(COMM,\
                         subm_regex_yes = covid_regex,\
                         comm_regex_yes = fake_regex,\
                         subm_regex_no = fake_regex,\
                         comm_regex_no = fake_regex_no)
CORONAFLAG.head()

Assign variable to one that has flag

In [ ]:
### This does not work yet
COMM['has_flag'] = False
COMM.loc[CORONAFLAG.index,'has_flag'] = True
Counter(COMM['has_flag'])

In [ ]:
Counter(COMM.subreddit[COMM.has_flag])

### Getting flags using regex

In [ ]:
%%capture --no-stdout  --no-display
print_corona_sc(CORONAFLAG, 100, charlim = 500)

## Find keywords for submissions

In [ ]:
S = pd.read_csv("http://ec2-3-8-1-23.eu-west-2.compute.amazonaws.com/vlad/R_all_submissions.csv")
S = pd.merge(S, ALL03[['subm_id','date_created','date_added']].rename(columns = {'subm_id':'id'}), on = 'id')
S['id']='t3_'+ S['id']
S['domain']= [extract_domain(url) for url in S['url']]
S.columns = S.columns.str.lstrip()

In [ ]:
import spacy
nlp = spacy.load('en_core_web_sm')
docs = [nlp(text) for text in S.title]

In [ ]:
chunks = [chunk.text.lower() for doc in docs for chunk in doc.noun_chunks if not chunk.root.is_stop and not chunk.root.is_punct]
common_chunks = Counter(chunks).most_common(300)
with open('submissions_covid_topwords.txt', 'w') as the_file:
    for word, n in common_chunks:
        the_file.write(word + ', ' + str(n)+'\n')

In [ ]:
ventilators, lockdown, crisis, quarantine, virus, masks, outbreak, hospitals, lockdown, deaths, spread, social distancing, izolation, medical supplies, patients, treatment, vaccine, cdc

## Find keywords for comments

In [ ]:
import spacy
nlp = spacy.load('en_core_web_sm')

In [ ]:
def extract_pos(text, df=False):
    pos_list = [(t.text, t.lemma_, t.pos_, t.tag_, t.dep_, list(t.ancestors),list(t.children)) for t in nlp(text)]
    if df:
        return(pd.DataFrame(pos_list, columns=['text','lemma','pos','tag','dep','ancestors','children']))   
    else:
        return(pos_list)

Example

In [ ]:
extract_pos("You are spreading fake news.")

In [ ]:
def children_in(tok, wordlist):
    value = False
    for a in tok.children:
        if a.lower_ in wordlist:
            value = a.lower_
            break
    return(value)

In [ ]:
def fake_sent_pos(sent):
    v,s,o,neg = [False]*4
    final_phrase = False
    for tok in sent:
        if tok.dep_ == "ROOT":
            if tok.lemma_ in predicates:            
                v = tok.text
                for c in tok.children:
                    if (c.dep_ == "neg"):
                        neg = True
                    elif (c.dep_ == "nsubj") and (c.lower_ in subjects):
                        s = c.text
                    elif c.dep_ in ["dobj","attr","acomp"]:
                        if (c.lower_ in fullobjects):                            
                            o = c.text
                        elif (c.lower_ in partialobjects) and children_in(c, partialattributes):
                            o = children_in(c, partialattributes) + ' ' + c.text
                    elif c.lower_ == 'like' and c.dep_ == "prep":
                        for like in c.children:
                            if (like.lower_ in fullobjects):                            
                                o = 'like ' + like.text
                            elif (like.lower_ in partialobjects) and children_in(like, partialattributes):
                                o = 'like ' + children_in(like, partialattributes) + ' ' + like.text
                            if o and not s:
                                s=''
                if v and o and not neg:
                    if s:
                        final_phrase = '%s %s %s' % (s, v, o)
                    elif s=='':
                        final_phrase = '%s %s' % (v, o)
            elif tok.pos_ in ['NOUN','PROPN']:
                if tok.lower_ in fullobjects:
                    final_phrase = tok.text
                elif tok.lower_ in partialobjects and children_in(tok, partialattributes):
                    final_phrase = children_in(tok, partialattributes) + ' ' + tok.lower_
    return(final_phrase) 

In [ ]:
CORONASUB = filter_comm(COMM,\
                         subm_regex_yes = covid_regex,\
                         comm_regex_yes = '.*',\
                         subm_regex_no = fake_regex,\
                         comm_regex_no = '###notatext###')
CORONASUB.loc[CORONASUB.index,'flag_pos']=None
#CORONASUB.loc[CORONASUB.index,'flag_sent']=None
CORONASUB = CORONASUB.sort_values('date_created', ascending = False)
CORONASUB.shape

### Define keywords in all syntactic groups

In [ ]:
subjects = ['you','it','this','article','title','headline']
predicates = ['be','spread','stop','call',\
              'look','sound','feel','seem','smell']
fullobjects = ['disinformation','misinformation','misleading','bullshit','bs','falsehood','fabrication']
partialobjects = ['news','information','title','source','article']
partialattributes = ['fake','false','misleading','untrustworthy','bogus','bullshit']
###
fake_regex_wrds = 'fake|false\\w*|bogus|'+\
                  'mislead\\w*|disinf\\w*|misinf\\w*|malinf\\w*|'+\
                  'bullshit|\\bbs\\b|'+\
                  'news|info\\w*|title|article|source|submission'

### Run for each row POS detection

In [ ]:
n, iterr = [0,0]
subm_index_old, comm_index_old = ['','']
found_sentences = []
for index, row in CORONASUB.iterrows():
    iterr+=1
    for sent in nlp(str(row.comm_body)).sents:
        fake_pos = fake_sent_pos(sent)
        if fake_pos:
            n+=1        
            if index[0]!=subm_index_old:
                print('\n\n\n')
                print("%d / %d" % (n, iterr))
                print(row.subreddit)
                print('--------------------')
                highlight_re('<b>'+row.subm_title+'</b>', covid_regex)
                print('URL: %s \nPublished: %s ' % (row.subm_url, row.date_created[:20]))
                subm_index_old = index[0]
            if index[1]!=comm_index_old:
                highlight_re(row.comm_body, fake_regex_wrds)
                comm_index_old = index[1]
            CORONASUB.loc[index,'flag_pos'] = fake_pos
            found_sentences.append((index[0], index[1], sent, fake_pos))
            print('Q: ' + fake_pos)
    if iterr % 50 == 1:
        print("#", end = '')
print("%d found / %d searched" % (n, iterr))

In [ ]:
found_sentences_df = pd.DataFrame([(a,b,c.text,d) for a,b,c,d in found_sentences],\
                                  columns = ['subm_id','comm_id','sentence','structure'])
found_sentences_df.to_csv("found_sentences_pos.csv", index = False)

In [ ]:
Counter(CORONASUB.flag_pos).most_common(10)

In [ ]:
Counter(CORONASUB.subreddit[~CORONASUB.flag_pos.isna()]).most_common()

### Percent sources most often flagged 

Filter: source mentioned at least 10 times and flagged at least 2 times

In [ ]:
CORONASUB.subm_title[CORONASUB.has_flag_pos]

In [ ]:
CORONASUB['domain'] = CORONASUB.subm_url.apply(extract_domain)
CORONASUB['has_flag_pos'] = ~CORONASUB.flag_pos.isna()
DOMAINS = CORONASUB.groupby('domain').agg(count_all = ("has_flag_pos","count"), count_flag = ("has_flag_pos","sum"))
DOMAINS['percent_flag'] = 100*DOMAINS.count_flag/DOMAINS.count_all

In [ ]:
DOMAINS.query('count_all>9 & count_flag>1').sort_values('percent_flag', ascending = False).head(30)